# Práctica: el algoritmo de Grover

Esta práctica construye Grover pieza por pieza, con Qiskit y el simulador Aer:

| Parte | Tema |
|---|---|
| 0 | Preparación y herramientas |
| 1 | La compuerta Z multicontrolada (MCZ) |
| 2 | ¿Qué es un oracle? |
| 3 | El difusor y una iteración de Grover |
| 4 | Jugando con el número de iteraciones $t$ |
| 5 | Varios estados marcados... o ninguno |
| 6 | **Playground**: estados marcados con el oracle builder de IBM |
| 7 | Reto: oracles de verdad |

Los ejercicios se escriben donde dice `# TU CÓDIGO AQUÍ`. Muchos se corrigen solos (`check_oracle`, `check_t`,
`check_secret`) y todos traen la solución plegada en **Ver solución**.

**Ejecuta las celdas en orden**: cada parte usa funciones definidas en las anteriores.

---
# Parte 0 · Preparación

Si esta celda falla con `ModuleNotFoundError`, el notebook está usando un Python que no tiene las dependencias.
En VS Code: **Select Kernel → Python Environments →** `.venv` (uv) o `grover-practica` (conda). En Jupyter Lab,
ábrelo con `uv run jupyter lab` o desde el entorno conda activado.

In [ ]:
%matplotlib inline
import math
import random
import sys

import numpy as np
import matplotlib.pyplot as plt
import qiskit
import qiskit_aer
from qiskit import ClassicalRegister, QuantumCircuit, transpile
from qiskit.circuit.library import MCMTGate, MCXGate, ZGate, grover_operator
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

print("Python    ", sys.version.split()[0], "  ", sys.executable)
print("qiskit    ", qiskit.__version__)
print("qiskit-aer", qiskit_aer.__version__)

### Herramientas

La siguiente celda define funciones de apoyo: simular, graficar y autocorregir. **No hace falta leerla ahora**;
las piezas importantes de Grover (MCZ, oracles, difusor) las construyes tú a lo largo de la práctica.

Convención de bits (la de Qiskit): el string `"q_{n-1} … q_1 q_0"`. **El qubit 0 es el carácter de la derecha.**

In [ ]:
# ------------------------------------------------------------------ simulación
SIM = AerSimulator()


def statevector(qc):
    """Vector de estado exacto (se ignoran las mediciones finales)."""
    return Statevector(qc.remove_final_measurements(inplace=False))


def amplitudes(qc, n=None):
    """Amplitudes de los 2^n estados de búsqueda (las ancillas, si hay, deben terminar en |0>)."""
    n = qc.num_qubits if n is None else n
    return statevector(qc).data[: 2**n]


def probabilities(qc, n=None):
    """Probabilidad exacta de cada string de n bits."""
    n = qc.num_qubits if n is None else n
    probs = statevector(qc).probabilities(qargs=list(range(n)))
    return {format(i, f"0{n}b"): float(p) for i, p in enumerate(probs)}


def simulated_success(qc, marked, n=None):
    """Probabilidad exacta de medir alguno de los estados marcados."""
    probs = probabilities(qc, n)
    return sum(probs[s] for s in set(marked))


def counts(qc, shots=1024, n=None, seed=None):
    """Ejecuta en AerSimulator. Si el circuito no mide, se miden los primeros n qubits."""
    if qc.num_clbits == 0:
        n = qc.num_qubits if n is None else n
        qc = qc.copy()
        creg = ClassicalRegister(n, "meas")
        qc.add_register(creg)
        qc.measure(range(n), creg)
    result = SIM.run(transpile(qc, SIM), shots=shots, seed_simulator=seed).result()
    return dict(result.get_counts())


# ------------------------------------------------------------------ fórmulas (Partes 3 y 4)
def theta(N, M):
    """sin θ = sqrt(M/N)."""
    return math.asin(math.sqrt(M / N))


def optimal_t(N, M):
    """Entero más cercano a π/(4θ) - 1/2, que es floor(π/(4θ)). Con M = 0, t = 0."""
    th = theta(N, M)
    return 0 if th == 0 else math.floor(math.pi / (4 * th))


def success_probability(N, M, t):
    """P(medir un marcado tras t iteraciones) = sin²((2t+1)θ)."""
    return math.sin((2 * t + 1) * theta(N, M)) ** 2


# ------------------------------------------------------------------ autocorrección
def oracle_block(qc, n=None):
    """Matriz sobre los n qubits de búsqueda con las ancillas en |0> (comprueba que vuelvan a |0>)."""
    n = qc.num_qubits if n is None else n
    block = Operator(qc).data[: 2**n, : 2**n]
    if not np.allclose(block.conj().T @ block, np.eye(2**n), atol=1e-8):
        raise AssertionError("Las ancillas no vuelven a |0>: falta deshacer (uncompute) los cálculos auxiliares.")
    return block


def show_phases(qc, n=None):
    """Imprime la fase (+1 / -1) de cada estado de la base y devuelve los que tienen -1."""
    n = qc.num_qubits if n is None else n
    block = oracle_block(qc, n)
    if not np.allclose(block - np.diag(np.diag(block)), 0, atol=1e-8):
        print("Ojo: la matriz no es diagonal, así que no es un oracle de fase.")
    negative = []
    for i, d in enumerate(np.diag(block)):
        s = format(i, f"0{n}b")
        sign = "-1" if np.isclose(d, -1) else "+1" if np.isclose(d, 1) else f"{complex(d):.3f}"
        if sign == "-1":
            negative.append(s)
        if n <= 4 or sign != "+1":
            print(f"|{s}⟩ : {sign}")
    if n > 4:
        print(f"(los otros {2**n - len(negative)} estados tienen fase +1)")
    return negative


def check_oracle(qc, marked, n=None):
    """Comprueba que qc sea el oracle de fase del conjunto `marked`."""
    marked = list(marked)
    n = qc.num_qubits if n is None else n
    try:
        block = oracle_block(qc, n)
    except AssertionError as err:
        print(f"❌ {err}")
        return False
    if not np.allclose(block - np.diag(np.diag(block)), 0, atol=1e-8):
        print("❌ No es diagonal: tu circuito mueve amplitud entre estados.")
        print("   Un oracle de fase solo cambia signos, nunca cambia |x> por otro |y>.")
        return False
    got = np.real_if_close(np.diag(block))
    want = np.ones(2**n)
    for s in set(marked):
        want[int(s, 2)] = -1
    wrong = [format(i, f"0{n}b") for i in range(2**n) if not np.isclose(got[i], want[i])]
    if not wrong:
        print(f"✅ Correcto: el oracle marca exactamente {sorted(set(marked)) or 'ningún estado'}.")
        return True
    if np.allclose(got, -want):
        print("⚠️  Coincide salvo una fase global -1 (marca justo lo contrario). Revisa la construcción.")
        return False
    print(f"❌ Fase incorrecta en: {wrong}")
    missing = [s for s in wrong if s in marked]
    extra = [s for s in wrong if s not in marked]
    if missing:
        print(f"   Deberían tener fase -1: {missing}")
    if extra:
        print(f"   No deberían estar marcados: {extra}")
    return False


def check_t(n, M, t):
    """Comprueba una predicción del número óptimo de iteraciones."""
    best = optimal_t(2**n, M)
    if t == best:
        print(f"✅ Correcto: con N = {2**n} y M = {M}, t_opt = {best}.")
        return True
    print(f"❌ Para N = {2**n} y M = {M} el óptimo no es {t}. Pista: t ≈ π/(4θ) - 1/2.")
    return False


# ------------------------------------------------------------------ gráficas
def show(qc, **kwargs):
    """Dibuja un circuito (cerrando la figura para que no salga dos veces)."""
    fig = qc.draw("mpl", **kwargs)
    display(fig)
    plt.close(fig)


def histograma(data, **kwargs):
    """plot_histogram de Qiskit, mostrado una sola vez."""
    fig = plot_histogram(data, **kwargs)
    display(fig)
    plt.close(fig)


SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
NORMAL, MARKED, TEXT, GRID = SERIES[0], SERIES[1], "#52514e", "#e5e4df"


def _style(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def plot_amplitudes(state, marked=(), n=None, title=None, show_mean=True, ax=None):
    """Barras con las amplitudes (con signo). Naranja = marcado; línea punteada = media."""
    if isinstance(state, Statevector):
        n = state.num_qubits if n is None else n
        amps = state.data[: 2**n]
    else:
        n = state.num_qubits if n is None else n
        amps = amplitudes(state, n)
    if not np.allclose(np.imag(amps), 0, atol=1e-9):
        raise ValueError("Las amplitudes tienen parte imaginaria; esta gráfica solo muestra amplitudes reales.")
    real = np.real(amps)
    labels = [format(i, f"0{n}b") for i in range(2**n)]
    marked = set(marked)
    if ax is None:
        _, ax = plt.subplots(figsize=(max(6, 0.32 * 2**n), 3.2))
    ax.bar(range(2**n), real, color=[MARKED if s in marked else NORMAL for s in labels], width=0.7)
    ax.axhline(0, color=TEXT, linewidth=0.8)
    handles = [plt.Rectangle((0, 0), 1, 1, color=NORMAL, label="no marcado")]
    if marked:
        handles.append(plt.Rectangle((0, 0), 1, 1, color=MARKED, label="marcado"))
    if show_mean:
        ax.axhline(real.mean(), color=TEXT, linestyle="--", linewidth=1)
        handles.append(plt.Line2D([], [], color=TEXT, linestyle="--", label="media"))
    ax.set_xticks(range(2**n))
    ax.set_xticklabels(labels, rotation=90 if n > 3 else 0, fontsize=8 if n > 4 else 9)
    ax.set_ylabel("amplitud")
    lim = max(1e-9, np.abs(real).max()) * 1.15
    ax.set_ylim(-lim, lim)
    if title:
        ax.set_title(title, fontsize=11)
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8, frameon=False)
    _style(ax)
    return ax


def plot_success_vs_t(N, Ms, t_max=None, simulated=None, highlight_t=None, ax=None):
    """P(éxito) = sin²((2t+1)θ) contra t. `simulated` = {M: [P(0), P(1), ...]}."""
    Ms = [Ms] if isinstance(Ms, int) else list(Ms)
    if t_max is None:
        t_max = max(3, 3 * max(optimal_t(N, M) for M in Ms) + 2)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 3.4))
    tt = np.linspace(0, t_max, 400)
    ts = np.arange(t_max + 1)
    for i, M in enumerate(Ms):
        c = SERIES[i % len(SERIES)]
        ax.plot(tt, np.sin((2 * tt + 1) * theta(N, M)) ** 2, color=c, linewidth=1, alpha=0.35)
        ax.plot(ts, [success_probability(N, M, t) for t in ts], "o", color=c, markersize=6, label=f"M = {M} (teoría)")
        if simulated and M in simulated:
            ax.plot(range(len(simulated[M])), simulated[M], "x", color=TEXT, markersize=7, label=f"M = {M} (simulación)")
        if M:
            ax.axvline(optimal_t(N, M), color=c, linestyle=":", linewidth=1)
    if highlight_t is not None:
        ax.axvspan(highlight_t - 0.3, highlight_t + 0.3, color=GRID, alpha=0.8, zorder=0, label=f"t = {highlight_t}")
    ax.set_xlabel("t (iteraciones de Grover)")
    ax.set_ylabel("P(medir un marcado)")
    ax.set_ylim(-0.03, 1.05)
    ax.set_xticks(ts if t_max <= 20 else np.linspace(0, t_max, 11).astype(int))
    ax.set_title(f"N = {N}: las líneas punteadas marcan t_opt", fontsize=11)
    ax.legend(fontsize=8, frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
    _style(ax)
    return ax


def plot_geometry(N, M, t, ax=None):
    """Cada iteración rota el estado 2θ hacia |w>. Horizontal: |w⊥>; vertical: |w>."""
    th = theta(N, M)
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 4.2))
    circle = np.linspace(-math.pi, math.pi, 400)
    ax.plot(np.cos(circle), np.sin(circle), color=GRID, linewidth=1)
    ax.axhline(0, color=GRID, linewidth=1)
    ax.axvline(0, color=GRID, linewidth=1)
    for k in range(t + 1):
        ang, last = (2 * k + 1) * th, k == t
        ax.annotate("", xy=(math.cos(ang), math.sin(ang)), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=MARKED if last else NORMAL,
                                    lw=2 if last else 1, alpha=1 if last else 0.45))
        ax.text(1.08 * math.cos(ang), 1.08 * math.sin(ang), f"t={k}", fontsize=8, color=TEXT, ha="center", va="center")
    ax.text(1.12, -0.08, "|w⊥⟩", color=TEXT, fontsize=10)
    ax.text(0.04, 1.1, "|w⟩", color=TEXT, fontsize=10)
    ax.set_xlim(-1.25, 1.35)
    ax.set_ylim(-1.25, 1.3)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_title(f"θ = {math.degrees(th):.1f}°, t = {t}: P = {success_probability(N, M, t):.3f}", fontsize=10)
    return ax


print("Herramientas listas.")

Prueba rápida: un estado de Bell en Aer. Deben salir `00` y `11` mitad y mitad.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
print(counts(bell, shots=1000))

---
# Parte 1 · La compuerta Z multicontrolada (MCZ)

**Objetivos:** entender qué hace `MCMTGate(ZGate(), n-1, 1)` (pone un signo $-1$ a **un solo** estado,
$|1\dots1\rangle$), leer bien los bitstrings de Qiskit y usar X ("controles abiertos") para marcar **cualquier** estado.

### 1.1 Empezamos por Z

$$Z = \begin{pmatrix}1 & 0\\ 0 & -1\end{pmatrix},\qquad Z|0\rangle = |0\rangle,\quad Z|1\rangle = -|1\rangle.$$

Z no cambia **qué** estado tienes, solo le pone un signo (una fase) a $|1\rangle$. Si mides en la base
computacional, no notas la diferencia. Esa es la idea de todo oracle de fase.

In [ ]:
z = QuantumCircuit(1)
z.z(0)
print(np.real(Operator(z).data))

### 1.2 CZ: la Z controlada

`cz(0, 1)` pone un $-1$ solo cuando **ambos** qubits valen 1: $CZ = \mathrm{diag}(1, 1, 1, -1)$.
No importa cuál llamas "control" y cuál "objetivo": `cz(0, 1)` y `cz(1, 0)` son la misma matriz.

In [ ]:
a = QuantumCircuit(2); a.cz(0, 1)
b = QuantumCircuit(2); b.cz(1, 0)
print(np.real(np.diag(Operator(a).data)))
print("¿cz(0,1) == cz(1,0)?", Operator(a) == Operator(b))

### 1.3 La MCZ de n qubits

`MCMTGate(ZGate(), n-1, 1)` es una Z con $n-1$ controles: pone $-1$ solo en $|1\dots1\rangle$.

$$\mathrm{MCZ}_n = I - 2\,|1\dots1\rangle\langle 1\dots1|.$$

Al dibujarla, Qiskit le pone la etiqueta **`1-Z`**: es esta misma compuerta (una Z sobre 1 qubit objetivo, con
$n-1$ controles). La guardamos en una función `mcz(n)` para el resto de la práctica. `show_phases` imprime la fase que el
circuito le pone a cada estado de la base.

In [ ]:
def mcz(n):
    """Z multicontrolada sobre n qubits: un único -1 en |1...1>."""
    qc = QuantumCircuit(n, name=f"MCZ({n})")
    if n == 1:
        qc.z(0)
    else:
        qc.compose(MCMTGate(ZGate(), n - 1, 1), inplace=True)
    return qc


show(mcz(4))
show_phases(mcz(4));

**Pregunta.** ¿Cuántos estados de la base tiene un registro de 4 qubits? ¿A cuántos les cambió la fase la MCZ?

### 1.4 ¡Cuidado con el orden de los bits!

Qiskit escribe los estados como $|q_{n-1}\dots q_1 q_0\rangle$: **el qubit 0 es el de la derecha**.
Si aplicamos X solo al qubit 0 de un registro de 3 qubits obtenemos `001`, no `100`.

In [ ]:
qc = QuantumCircuit(3)
qc.x(0)
print({k: v for k, v in probabilities(qc).items() if v > 0})

### 1.5 Controles abiertos: marcar cualquier estado

La MCZ solo marca $|1\dots1\rangle$. Para marcar otro estado $|w\rangle$:

1. aplica X a los qubits donde $w$ tiene un **0** (así $|w\rangle \to |1\dots1\rangle$),
2. aplica la MCZ (pone el $-1$),
3. vuelve a aplicar las mismas X (deshace el cambio).

$$O_w = X^{\otimes a}\,\mathrm{MCZ}\,X^{\otimes a} = I - 2|w\rangle\langle w|.$$

Ejemplo con $w = $ `"010"`: los ceros están en los qubits 0 y 2.

In [ ]:
qc = QuantumCircuit(3)
qc.x([0, 2])
qc.compose(mcz(3), inplace=True)
qc.x([0, 2])
show(qc)
show_phases(qc);

Lo convertimos en funciones: `zero_indices(w)` dice dónde hay ceros y `mark_state(w)` construye el oracle.

In [ ]:
def zero_indices(w):
    """Qubits que valen 0 en el string w (el qubit 0 es el carácter de la derecha)."""
    return [q for q, bit in enumerate(reversed(w)) if bit == "0"]


def mark_state(w):
    """Oracle de un solo estado: I - 2|w><w|."""
    n = len(w)
    qc = QuantumCircuit(n, name=f"O_{w}")
    zeros = zero_indices(w)
    if zeros:
        qc.x(zeros)
    qc.compose(mcz(n), inplace=True)
    if zeros:
        qc.x(zeros)
    return qc


print("ceros de '010' en los qubits:", zero_indices("010"))
print("¿mi circuito == mark_state('010')?", Operator(qc) == Operator(mark_state("010")))

### 1.6 ¿Qué le hace el oracle a una superposición?

Con $H^{\otimes n}$ preparamos $|s\rangle$, la superposición uniforme. Después del oracle todas las amplitudes
siguen valiendo $1/\sqrt{N}$ en magnitud, pero la del estado marcado cambió de signo.

In [ ]:
qc = QuantumCircuit(3)
qc.h(range(3))
qc.compose(mark_state("010"), inplace=True)
plot_amplitudes(qc, marked=["010"], title="Después de H y del oracle");

### Ejercicios de la Parte 1

**Ejercicio 1.1.** Construye **a mano** (con `x` y `mcz`, sin `mark_state`) el oracle que marca `"0101"` en 4 qubits.

In [ ]:
ej11 = QuantumCircuit(4)
# TU CÓDIGO AQUÍ


check_oracle(ej11, ["0101"]);

<details>
<summary><b>Ver solución</b></summary>

`"0101"` tiene ceros en los qubits 1 y 3 (leyendo de derecha a izquierda):

```python
ej11 = QuantumCircuit(4)
ej11.x([1, 3])
ej11.compose(mcz(4), inplace=True)
ej11.x([1, 3])
```

</details>

**Ejercicio 1.2.** Marca `"110"` en 3 qubits. ¿Cuántas compuertas X necesitas?

In [ ]:
ej12 = QuantumCircuit(3)
# TU CÓDIGO AQUÍ


check_oracle(ej12, ["110"]);

<details>
<summary><b>Ver solución</b></summary>

Un solo cero, en el qubit 0: dos X en total (una antes y otra después).

```python
ej12 = QuantumCircuit(3)
ej12.x(0)
ej12.compose(mcz(3), inplace=True)
ej12.x(0)
```

</details>

**Ejercicio 1.3.** ¿Qué pasa si pones las X en los qubits *leídos de izquierda a derecha*? Construye ese circuito
erróneo para `"110"` (X en el qubit 2) y usa `show_phases` para ver qué estado marcó.

In [ ]:
mal = QuantumCircuit(3)
# TU CÓDIGO AQUÍ


show_phases(mal);

<details>
<summary><b>Ver solución</b></summary>

```python
mal = QuantumCircuit(3)
mal.x(2)
mal.compose(mcz(3), inplace=True)
mal.x(2)
show_phases(mal)   # marca "011": el string al revés
```
Es el error más común con Qiskit: el oracle marca el bitstring invertido.

</details>

**Ejercicio 1.4 (para pensar).** ¿Qué circuito marca $|0\dots0\rangle$? Aparece de nuevo en la Parte 3, dentro del difusor.

<details>
<summary><b>Ver solución</b></summary>

X en **todos** los qubits, MCZ, X en todos los qubits. Es `mark_state("000")`.

</details>

---
# Parte 2 · ¿Qué es un oracle?

**Objetivos:** entender el oracle como una **caja negra** que *reconoce* soluciones sin decir cuáles son;
construir oracles para varios estados, para ninguno y a partir de una función; ver que **solo cambia fases**.

### 2.1 La idea

Tenemos $f:\{0,1\}^n \to \{0,1\}$. Los $x$ con $f(x) = 1$ son las **soluciones** (estados marcados).
Verificar si un $x$ es solución es fácil; *encontrar* uno, difícil. Piensa en un candado: probar una
combinación es inmediato, adivinar la correcta no.

El **oracle de fase** es la versión cuántica de "probar una combinación":

$$U_f\,|x\rangle = (-1)^{f(x)}\,|x\rangle.$$

- Es una matriz **diagonal** con entradas $\pm 1$.
- Grover solo interactúa con el problema a través de $U_f$: cada aplicación cuenta como **una consulta**.
- El oracle "sabe" cuáles son las soluciones, pero no las revela al medir: solo cambia signos.

### 2.2 Varios estados marcados

Para marcar un conjunto $W$ se aplica un `mark_state` por elemento:

$$U_f = \prod_{w \in W} \left(I - 2|w\rangle\langle w|\right) = I - 2\sum_{w\in W} |w\rangle\langle w|.$$

Como todas son diagonales, **conmutan**: el orden no importa.

In [ ]:
def phase_oracle(marked, n=None):
    """Oracle de fase para una lista de estados. Con lista vacía hay que dar n."""
    if isinstance(marked, str):
        marked = [marked]
    marked = list(marked)
    n = len(marked[0]) if n is None else n
    qc = QuantumCircuit(n, name="Oracle")
    for w in marked:
        qc.compose(mark_state(w), inplace=True)
    return qc


oracle = phase_oracle(["011", "110"])
show(oracle)
show_phases(oracle)
print("¿El orden importa?", not (Operator(oracle) == Operator(phase_oracle(["110", "011"]))))

#### Trampa: marcar dos veces es no marcar

Si el mismo estado aparece dos veces, su fase se multiplica por $(-1)^2 = +1$: las dos MCZ se cancelan.

In [ ]:
show_phases(phase_oracle(["101", "101"]));

**Pregunta.** ¿Y si `"101"` aparece tres veces? Predice y compruébalo.

### 2.3 Casos extremos: ningún estado y todos

- **Ningún estado marcado** ($f \equiv 0$): $U_f = I$. El oracle no hace nada.
- **Todos marcados** ($f \equiv 1$): $U_f = -I$, una **fase global**: físicamente es lo mismo que no hacer nada.

In [ ]:
vacio = phase_oracle([], n=3)
print("oracle vacío == identidad:", Operator(vacio) == Operator(QuantumCircuit(3)))

todos = phase_oracle([format(x, "03b") for x in range(8)])
print("diagonal de 'todos':", np.real(np.diag(Operator(todos).data)))
print("¿igual a la identidad salvo fase global?", Operator(todos).equiv(Operator(QuantumCircuit(3))))

### 2.4 Oracle a partir de una función

Cualquier función booleana da un oracle. Por ejemplo, "$x$ es múltiplo de 3" con 4 bits:

In [ ]:
def marked_from_function(f, n):
    """Strings x de n bits con f(x) verdadero."""
    return [format(x, f"0{n}b") for x in range(2**n) if f(x)]


def oracle_from_function(f, n):
    """U_f|x> = (-1)^f(x)|x>, evaluando f clásicamente (útil para aprender)."""
    return phase_oracle(marked_from_function(f, n), n=n)


print("marcados:", marked_from_function(lambda x: x % 3 == 0, 4))
show_phases(oracle_from_function(lambda x: x % 3 == 0, 4));

> **Ojo.** `oracle_from_function` evalúa $f$ clásicamente para construir el circuito. Sirve para aprender, pero es
> hacer trampa: si ya evaluaste $f$ en todos los $x$, ya conoces las soluciones. Un oracle de verdad **calcula** $f$
> con compuertas reversibles. Eso lo haces en la Parte 7.

### 2.5 El oracle solo cambia fases

Comparemos las probabilidades **antes** y **después** del oracle:

In [ ]:
W = ["011", "110"]
antes = QuantumCircuit(3); antes.h(range(3))
despues = antes.copy(); despues.compose(phase_oracle(W), inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
plot_amplitudes(antes, W, title="Antes del oracle", ax=axes[0])
plot_amplitudes(despues, W, title="Después del oracle", ax=axes[1])
plt.tight_layout(); plt.show()

print("P antes  :", {k: round(v, 3) for k, v in probabilities(antes).items()})
print("P después:", {k: round(v, 3) for k, v in probabilities(despues).items()})

Son **idénticas**: $|{-a}|^2 = |a|^2$. Si midiéramos justo después del oracle no aprenderíamos nada. Para convertir
la fase en algo medible hace falta que las amplitudes **interfieran**: ese es el trabajo del **difusor** (Parte 3).

### 2.6 Oracle de bit-flip y *phase kickback*

Muchos libros definen el oracle con un qubit extra: $B_f\,|x\rangle|y\rangle = |x\rangle\,|y \oplus f(x)\rangle$.
Es la forma natural de un circuito clásico reversible (Toffolis). Si la ancilla está en $|-\rangle$:

$$B_f\,|x\rangle|-\rangle = (-1)^{f(x)}\,|x\rangle|-\rangle.$$

La ancilla no cambia y la fase "rebota" hacia $|x\rangle$. Los dos tipos de oracle son equivalentes.

In [ ]:
def bitflip_oracle(marked, n=None):
    """|x>|y> -> |x>|y XOR f(x)>, con la ancilla y en el qubit n."""
    marked = list(marked)
    n = len(marked[0]) if n is None else n
    qc = QuantumCircuit(n + 1, name="B_f")
    for w in marked:
        zeros = zero_indices(w)
        if zeros:
            qc.x(zeros)
        qc.append(MCXGate(n), list(range(n + 1)))
        if zeros:
            qc.x(zeros)
    return qc


def phase_via_kickback(marked, n=None):
    """Oracle de fase hecho con B_f y la ancilla en |->. La ancilla entra y sale en |0>."""
    bf = bitflip_oracle(marked, n)
    n = bf.num_qubits - 1
    qc = QuantumCircuit(n + 1, name="O_kickback")
    qc.x(n); qc.h(n)
    qc.compose(bf, inplace=True)
    qc.h(n); qc.x(n)
    return qc


W = ["010", "111"]
show(phase_via_kickback(W))
check_oracle(phase_via_kickback(W), W, n=3);

### 2.7 Atajo: una MCZ más pequeña marca muchos estados

Si la condición solo depende de algunos qubits, basta una Z controlada sobre esos qubits. Por ejemplo,
$f(x) = q_0 \wedge \lnot q_2$ en 3 qubits: X en $q_2$, CZ entre $q_0$ y $q_2$, X en $q_2$. El qubit 1 no participa,
así que se marcan **dos** estados.

In [ ]:
atajo = QuantumCircuit(3)
atajo.x(2)
atajo.cz(0, 2)
atajo.x(2)
show_phases(atajo);

### Ejercicios de la Parte 2

**Ejercicio 2.1.** Construye el oracle que marca `{"0001", "1000", "1111"}`.

In [ ]:
ej21 = QuantumCircuit(4)
# TU CÓDIGO AQUÍ


check_oracle(ej21, ["0001", "1000", "1111"]);

<details>
<summary><b>Ver solución</b></summary>

```python
ej21 = QuantumCircuit(4)
for w in ["0001", "1000", "1111"]:
    ej21.compose(mark_state(w), inplace=True)
```
O directamente `ej21 = phase_oracle(["0001", "1000", "1111"])`.

</details>

**Ejercicio 2.2.** Usa `oracle_from_function` para marcar los **primos** de 4 bits (2, 3, 5, 7, 11, 13).

In [ ]:
def es_primo(x):
    # TU CÓDIGO AQUÍ
    return False

check_oracle(oracle_from_function(es_primo, 4), ["0010", "0011", "0101", "0111", "1011", "1101"]);

<details>
<summary><b>Ver solución</b></summary>

```python
def es_primo(x):
    return x > 1 and all(x % d for d in range(2, int(x**0.5) + 1))
```

</details>

**Ejercicio 2.3.** Con **una sola** Z controlada (más las X que necesites), marca todos los estados de 4 qubits con
$q_3 = 1$ y $q_0 = 0$. ¿Cuántos son?

In [ ]:
ej23 = QuantumCircuit(4)
# TU CÓDIGO AQUÍ


check_oracle(ej23, [format(x, "04b") for x in range(16) if (x >> 3) & 1 and not x & 1]);

<details>
<summary><b>Ver solución</b></summary>

Son 4 estados ($q_1$ y $q_2$ libres):

```python
ej23 = QuantumCircuit(4)
ej23.x(0)
ej23.cz(0, 3)
ej23.x(0)
```

</details>

**Ejercicio 2.4.** Este circuito **no** es un oracle de fase. Ejecuta `check_oracle` y explica por qué.

In [ ]:
raro = QuantumCircuit(2)
raro.cx(0, 1)
check_oracle(raro, ["11"]);

<details>
<summary><b>Ver solución</b></summary>

`cx` intercambia $|01\rangle \leftrightarrow |11\rangle$: mueve amplitud entre estados (no es diagonal).
Un oracle de fase solo puede cambiar signos.

</details>

---
# Parte 3 · El difusor y una iteración de Grover

### 3.1 Inversión respecto a la media

Sea $|s\rangle = H^{\otimes n}|0\dots0\rangle$ y $|\psi\rangle = \sum_x a_x|x\rangle$ con media $\bar a = \frac1N\sum_x a_x$.
Como $\langle s|\psi\rangle = \sqrt N\,\bar a$:

$$D|\psi\rangle = (2|s\rangle\langle s| - I)|\psi\rangle = \sum_x (2\bar a - a_x)\,|x\rangle.$$

Cada amplitud se **refleja respecto a la media**. Una amplitud negativa (la marcada) queda muy por encima; las demás
bajan un poco.

### 3.2 Construcción con compuertas

$$2|s\rangle\langle s| - I = H^{\otimes n}\left(2|0\rangle\langle 0| - I\right)H^{\otimes n}.$$

Y $2|0\rangle\langle0| - I = -(I - 2|0\rangle\langle 0|)$ es, salvo un signo global, **el oracle que marca
$|0\dots0\rangle$** (ejercicio 1.4). Entonces el difusor es otro "oracle" visto en la base de las H:

$$D \simeq H^{\otimes n} X^{\otimes n}\,\mathrm{MCZ}\,X^{\otimes n} H^{\otimes n}.$$

In [ ]:
def diffuser(n):
    """H X MCZ X H = I - 2|s><s|, es decir 2|s><s| - I salvo una fase global -1."""
    qc = QuantumCircuit(n, name="Difusor")
    qc.h(range(n))
    qc.x(range(n))
    qc.compose(mcz(n), inplace=True)
    qc.x(range(n))
    qc.h(range(n))
    return qc


show(diffuser(3))
N = 8
s = np.full(N, 1 / np.sqrt(N))
D_teoria = 2 * np.outer(s, s) - np.eye(N)
print("¿diffuser(3) == 2|s><s| - I?   ", np.allclose(Operator(diffuser(3)).data, D_teoria))
print("¿diffuser(3) == -(2|s><s| - I)?", np.allclose(Operator(diffuser(3)).data, -D_teoria))
print("¿iguales salvo fase global?     ", Operator(diffuser(3)).equiv(Operator(D_teoria)))

El signo global no se puede medir, así que da igual. (Algunas implementaciones lo corrigen y otras no.)

### 3.3 Una iteración, paso a paso

$n = 3$ ($N = 8$), marcamos $w = $ `"101"`.

In [ ]:
w = "101"
paso1 = QuantumCircuit(3); paso1.h(range(3))
paso2 = paso1.copy(); paso2.compose(phase_oracle(w), inplace=True)
paso3 = paso2.copy(); paso3.compose(diffuser(3), inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.2))
plot_amplitudes(paso1, [w], title="1) H: superposición uniforme", ax=axes[0])
plot_amplitudes(paso2, [w], title="2) Oracle: signo del marcado", ax=axes[1])
plot_amplitudes(-1 * statevector(paso3), [w], n=3, title="3) Difusor: inversión respecto a la media", ax=axes[2])
plt.tight_layout(); plt.show()

(En el panel 3 multiplicamos por $-1$ para quitar la fase global del difusor.)

Las cuentas: $a = 1/\sqrt8 \approx 0.354$. Tras el oracle la media es $\bar a = \frac{7a - a}{8} = 0.75a$. El marcado pasa
a $2\bar a + a = 2.5a \approx 0.884$ y los demás a $2\bar a - a = 0.5a \approx 0.177$.

In [ ]:
a = 1 / np.sqrt(8)
print("predicción marcado:", round(2.5 * a, 3), "  otros:", round(0.5 * a, 3))
print("simulado           :", np.round(np.real(-amplitudes(paso3)), 3))
print(f"P(medir {w}) = {simulated_success(paso3, [w]):.4f}")

### 3.4 La imagen geométrica

Sean $|w\rangle$ (superposición de los marcados) y $|w^\perp\rangle$ (de los no marcados):

$$|s\rangle = \sin\theta\,|w\rangle + \cos\theta\,|w^\perp\rangle,\qquad \sin\theta = \sqrt{M/N}.$$

- El **oracle** refleja respecto a $|w^\perp\rangle$.
- El **difusor** refleja respecto a $|s\rangle$.
- Dos reflexiones = una **rotación** de $2\theta$ hacia $|w\rangle$.

Tras $t$ iteraciones el ángulo es $(2t+1)\theta$, y por eso $P(\text{éxito}) = \sin^2\big((2t+1)\theta\big)$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for t, ax in zip([0, 1, 2], axes):
    plot_geometry(N=8, M=1, t=t, ax=ax)
plt.tight_layout(); plt.show()

### 3.5 El circuito completo

`grover_circuit(oracle, t)` pone las H y aplica $t$ veces $G = D\,O$. Si el oracle usa ancillas (Parte 7), se indica
con `n` cuántos qubits son de búsqueda; el difusor solo actúa sobre esos.

In [ ]:
def grover_iteration(oracle, n=None):
    """G = D · O sobre los primeros n qubits."""
    n = oracle.num_qubits if n is None else n
    qc = QuantumCircuit(oracle.num_qubits, name="G")
    qc.compose(oracle, inplace=True)
    qc.compose(diffuser(n), qubits=range(n), inplace=True)
    return qc


def grover_circuit(oracle, t, n=None, measure=False):
    """H^n seguido de t iteraciones de Grover (y medición de los n qubits de búsqueda)."""
    n = oracle.num_qubits if n is None else n
    g = grover_iteration(oracle, n)
    qc = QuantumCircuit(oracle.num_qubits, name=f"Grover t={t}")
    qc.h(range(n))
    for _ in range(t):
        qc.barrier()
        qc.compose(g, inplace=True)
    if measure:
        creg = ClassicalRegister(n, "meas")
        qc.add_register(creg)
        qc.barrier()
        qc.measure(range(n), creg)
    return qc


qc = grover_circuit(phase_oracle("101"), t=2, measure=True)
show(qc, fold=-1)
histograma(counts(qc, shots=2000, seed=7))

### Ejercicios de la Parte 3

**Ejercicio 3.1.** Con $n = 2$ y un marcado, haz a mano las cuentas de la inversión respecto a la media.
¿Cuánto vale la amplitud del marcado después de una iteración? Comprueba con el simulador.

In [ ]:
# TU CÓDIGO AQUÍ

<details>
<summary><b>Ver solución</b></summary>

$a = 1/2$. Tras el oracle: $(a, a, -a, a)$, media $\bar a = a/2$. El marcado queda en $2\bar a + a = 1$ y los demás
en $2\bar a - a = 0$. **Una iteración encuentra el marcado con probabilidad 1.**

```python
print(np.round(amplitudes(grover_circuit(phase_oracle("10"), t=1)), 3))   # [0, 0, -1, 0]
```
Sale $-1$ en lugar de $1$ por la fase global del difusor (sección 3.2); al medir da lo mismo.

</details>

**Ejercicio 3.2.** Construye a mano (sin `grover_circuit` ni `diffuser`) el circuito con $n = 3$ que busca `"011"`
con **dos** iteraciones y mide. Debe salir `011` alrededor del 94 % de las veces.

In [ ]:
ej32 = QuantumCircuit(3)
# TU CÓDIGO AQUÍ


ej32.measure_all()
histograma(counts(ej32, shots=2000))

<details>
<summary><b>Ver solución</b></summary>

```python
ej32 = QuantumCircuit(3)
ej32.h(range(3))
for _ in range(2):
    ej32.x(2); ej32.compose(mcz(3), inplace=True); ej32.x(2)            # oracle de "011"
    ej32.h(range(3)); ej32.x(range(3))                                  # difusor
    ej32.compose(mcz(3), inplace=True)
    ej32.x(range(3)); ej32.h(range(3))
ej32.measure_all()
```

</details>

**Ejercicio 3.3 (para pensar).** ¿Qué pasa si aplicas el difusor **sin** oracle a $|s\rangle$?

<details>
<summary><b>Ver solución</b></summary>

Todas las amplitudes valen $\bar a$, así que $2\bar a - a_x = a_x$: **no cambia nada** ($D|s\rangle = |s\rangle$).
Sin un oracle que rompa la simetría no hay amplificación. Lo retomamos en la Parte 5 (cero marcados).

</details>

---
# Parte 4 · Jugando con el número de iteraciones $t$

### 4.1 La fórmula

Con $N = 2^n$ estados y $M$ marcados, $\sin\theta = \sqrt{M/N}$ y tras $t$ iteraciones
$P(t) = \sin^2\big((2t+1)\theta\big)$. Queremos $(2t+1)\theta \approx \pi/2$, o sea $t \approx \frac{\pi}{4\theta} - \frac12$,
cuyo entero más cercano es

$$t_{\text{opt}} = \left\lfloor \frac{\pi}{4\theta} \right\rfloor \approx \frac{\pi}{4}\sqrt{\frac NM}.$$

Funciones: `theta(N, M)`, `optimal_t(N, M)`, `success_probability(N, M, t)` (definidas en la Parte 0).

In [ ]:
print(f"{'n':>2} {'N':>6} {'t_opt':>6} {'P(t_opt)':>9} {'clásico ~N/2':>13}")
for n in range(2, 13):
    N = 2**n
    t = optimal_t(N, 1)
    print(f"{n:>2} {N:>6} {t:>6} {success_probability(N, 1, t):>9.4f} {N // 2:>13}")

Cada iteración usa el oracle **una vez**. Con $n = 12$ Grover necesita ~50 consultas; una búsqueda clásica, en promedio, ~2048.

### 4.2 La oscilación: teoría contra simulación

$n = 4$, marcado `"1011"`, $t = 0, \dots, 12$:

In [ ]:
oracle = phase_oracle("1011")
sim = [simulated_success(grover_circuit(oracle, t), ["1011"]) for t in range(13)]
plot_success_vs_t(16, 1, t_max=12, simulated={1: sim});

La probabilidad **sube, llega al máximo en $t_{\text{opt}} = 3$ y vuelve a bajar**: el estado sigue girando y se pasa
de $|w\rangle$ (**sobre-rotación**). Es periódica, con periodo $\pi/(2\theta)$ en $t$. El segundo pico ($t = 9$) es incluso
un poco más alto, pero cuesta el triple de consultas; por eso se elige el **primer** máximo.

### 4.3 Lo que verías en un experimento (con shots)

In [ ]:
resultados = {f"t = {t}": counts(grover_circuit(oracle, t, measure=True), shots=2000, seed=1) for t in [1, 3, 6]}
histograma(list(resultados.values()), legend=list(resultados), figsize=(14, 4), bar_labels=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for t, ax in zip([1, 3, 6], axes):
    plot_geometry(16, 1, t, ax=ax)
plt.tight_layout(); plt.show()

Con $t = 6$ casi nunca sale `1011`: el vector giró de más y quedó cerca de $|w^\perp\rangle$.

### Ejercicios de la Parte 4

**Ejercicio 4.1.** Calcula con la fórmula (sin `optimal_t`) el número óptimo de iteraciones para $n = 4, 5, 6$ con un marcado.

In [ ]:
mi_t = {4: 0, 5: 0, 6: 0}   # TU RESPUESTA AQUÍ

for n, t in mi_t.items():
    check_t(n, 1, t)

<details>
<summary><b>Ver solución</b></summary>

$\theta = \arcsin(1/\sqrt N)$ y $t_{\text{opt}} = \lfloor \pi/(4\theta) \rfloor$:
$n=4$: $\theta = 14.48°$ → $t = 3$. $n=5$: $\theta = 10.18°$ → $t = 4$. $n=6$: $\theta = 7.18°$ → $t = 6$.

</details>

**Ejercicio 4.2.** Con $n = 2$, $t = 1$ da probabilidad **exactamente** 1. Explícalo con $\theta$.

<details>
<summary><b>Ver solución</b></summary>

$\sin\theta = 1/2 \Rightarrow \theta = 30°$ y $3 \cdot 30° = 90°$. Para $M = 1$ y $n \ge 3$ ningún entero cumple
$(2t+1)\theta = 90°$, así que $P(t_{\text{opt}}) < 1$ (aunque muy cerca).

</details>

**Ejercicio 4.3.** Para $n = 6$, ¿cada cuántas iteraciones se repite el máximo? Compáralo con $\pi/(2\theta)$.

In [ ]:
# TU CÓDIGO AQUÍ

<details>
<summary><b>Ver solución</b></summary>

```python
plot_success_vs_t(64, 1, t_max=30)
print(math.pi / (2 * theta(64, 1)))   # ≈ 12.5
```
Los máximos están en $t \approx 5.8,\ 18.3,\ 30.8$: cada ~12.5 iteraciones.

</details>

**Ejercicio 4.4.** Entre $n = 2, \dots, 10$ (con $M = 1$), ¿para qué $n$ es **más baja** $P(t_{\text{opt}})$?

In [ ]:
# TU CÓDIGO AQUÍ

<details>
<summary><b>Ver solución</b></summary>

```python
peor = min(range(2, 11), key=lambda n: success_probability(2**n, 1, optimal_t(2**n, 1)))
print(peor, success_probability(2**peor, 1, optimal_t(2**peor, 1)))
```
Es $n = 3$, con $P \approx 0.945$. Aun en el peor caso, Grover acierta más del 94 % de las veces.

</details>

---
# Parte 5 · Varios estados marcados... o ninguno

### 5.1 Más marcados, menos iteraciones

Con $M$ marcados, $\sin\theta = \sqrt{M/N}$: el ángulo inicial es mayor y se llega antes. Al medir sale **alguno** de los
marcados, cada uno con la misma probabilidad.

In [ ]:
for M in [1, 2, 3, 4]:
    print(f"M = {M}:  θ = {math.degrees(theta(16, M)):5.1f}°   t_opt = {optimal_t(16, M)}   "
          f"P(t_opt) = {success_probability(16, M, optimal_t(16, M)):.3f}")
plot_success_vs_t(16, [1, 2, 3, 4], t_max=10);

In [ ]:
W = ["0011", "0110", "1100"]
histograma(counts(grover_circuit(phase_oracle(W), t=optimal_t(16, 3), measure=True), shots=3000, seed=3))

### 5.2 $M = N/4$: una iteración exacta

$\sin\theta = 1/2 \Rightarrow \theta = 30°$ y $3\theta = 90°$: con un cuarto de los estados marcados, **una** iteración da
probabilidad 1.

In [ ]:
W = ["0000", "0101", "1010", "1111"]          # 4 de 16
print({k: round(v, 3) for k, v in probabilities(grover_circuit(phase_oracle(W), t=1)).items() if v > 1e-9})

### 5.3 $M = N/2$: Grover no ayuda

$\theta = 45°$, así que $(2t+1)\cdot 45°$ siempre es múltiplo impar de $45°$ y $P(t) = 1/2$ **para todo $t$**. Medir al
azar ya da 1/2.

In [ ]:
W = ["000", "011", "101", "110"]              # 4 de 8
print([round(simulated_success(grover_circuit(phase_oracle(W), t), W), 3) for t in range(6)])

### 5.4 $M > N/2$: iterar empeora

Con 6 de 8 marcados, $\theta = 60°$. Una iteración da $\sin^2(180°) = 0$: **nunca** sale un marcado.

In [ ]:
W = ["000", "001", "010", "011", "100", "101"]  # 6 de 8
for t in range(4):
    print(f"t = {t}:  P = {simulated_success(grover_circuit(phase_oracle(W), t), W):.3f}")

Con $t = 0$ (medir sin iterar) ya aciertas el 75 %. Regla práctica: si sospechas que $M > N/2$, no iteres, o agrega un
qubit para duplicar $N$ (ejercicio 5.3).

### 5.5 $M = 0$: no hay nada marcado

El oracle es la identidad, así que $G = D$, y $D|s\rangle = |s\rangle$ (salvo fase global). **El estado no se mueve**:
para cualquier $t$ la salida es uniforme.

In [ ]:
vacio = phase_oracle([], n=3)
for t in [0, 1, 2, 5]:
    print(f"t = {t}:", [round(v, 3) for v in probabilities(grover_circuit(vacio, t)).values()])
histograma(counts(grover_circuit(vacio, t=3, measure=True), shots=4000, seed=5))

**¿Cómo sé que no hay solución?** Grover siempre devuelve *algún* $x$. La receta es **verificar** $f(x)$ clásicamente. Si
tras varios intentos con distintos $t$ ningún $x$ cumple $f(x) = 1$, concluyes que, con alta probabilidad, no hay solución.

In [ ]:
def buscar(oracle, f, n, intentos=5, seed=0):
    rng = np.random.default_rng(seed)
    for k in range(intentos):
        t = int(rng.integers(0, optimal_t(2**n, 1) + 1))
        x = next(iter(counts(grover_circuit(oracle, t, measure=True), shots=1, seed=int(rng.integers(1_000_000)))))
        print(f"intento {k}: t = {t}, salió {x}, ¿f(x) = 1? {f(int(x, 2))}")
        if f(int(x, 2)):
            return x
    print("Ninguna solución encontrada: probablemente no existe.")

buscar(phase_oracle([], n=4), f=lambda x: False, n=4)

### 5.6 $M = N$: todo está marcado

El oracle es $-I$ (fase global): el estado tampoco se mueve, pero cualquier resultado es solución ($P = 1$).

In [ ]:
todos = [format(x, "03b") for x in range(8)]
print([round(simulated_success(grover_circuit(phase_oracle(todos), t), todos), 3) for t in range(4)])

### 5.7 Resumen

| $M/N$ | $\theta$ | Qué pasa |
|---|---|---|
| $0$ | $0°$ | El estado no se mueve; salida uniforme. Verificar clásicamente. |
| pequeño | $\approx\sqrt{M/N}$ | $t_{\text{opt}} \approx \frac\pi4\sqrt{N/M}$, $P \approx 1$. |
| $1/4$ | $30°$ | Una iteración, $P = 1$ exacto. |
| $1/2$ | $45°$ | $P = 1/2$ siempre: Grover no ayuda. |
| $> 1/2$ | $> 45°$ | Iterar empeora; mejor $t = 0$. |
| $1$ | $90°$ | Todo es solución. |

### 5.8 ¿Y si no conozco $M$?

Si eliges $t$ al azar entre $0$ y $T - 1$, con $T \ge 1/\sin 2\theta$, la probabilidad media de éxito es al menos $1/4$
(Boyer, Brassard, Høyer y Tapp, 1998). Su algoritmo va aumentando $T$ hasta encontrar una solución, sin conocer $M$,
con $O(\sqrt{N/M})$ consultas en promedio.

In [ ]:
for M in [1, 3, 10]:
    for T in [4, 8, 16]:
        print(f"M = {M:2d}, T = {T:2d}:  P media con t al azar = {np.mean([success_probability(64, M, t) for t in range(T)]):.3f}")

### Ejercicios de la Parte 5

**Ejercicio 5.1.** Con $n = 5$ y $M = 8$, predice $t_{\text{opt}}$ y compruébalo. Luego simula con 8 estados.

In [ ]:
check_t(5, 8, 0);   # TU RESPUESTA AQUÍ en lugar del 0

# TU CÓDIGO AQUÍ: simula con 8 estados marcados

<details>
<summary><b>Ver solución</b></summary>

$M/N = 1/4 \Rightarrow \theta = 30°$, $t_{\text{opt}} = 1$ y $P = 1$.

```python
check_t(5, 8, 1)
W = [format(x, "05b") for x in range(0, 32, 4)]
print(simulated_success(grover_circuit(phase_oracle(W), 1), W))
```

</details>

**Ejercicio 5.2.** Con $n = 4$, ¿con cuántos marcados **una** iteración da probabilidad exactamente 1? ¿Y exactamente 0?

In [ ]:
# TU CÓDIGO AQUÍ

<details>
<summary><b>Ver solución</b></summary>

```python
for M in range(17):
    print(M, round(success_probability(16, M, 1), 4))
```
$P = 1$ con $M = 4$ ($\theta = 30°$) y $P = 0$ con $M = 12$ ($\theta = 60°$).

</details>

**Ejercicio 5.3.** Con 3 qubits y 6 de 8 marcados (sección 5.4), agrega un **cuarto qubit** y marca los mismos 6 estados
solo cuando $q_3 = 0$. Ahora $N = 16$, $M = 6$. ¿Cuánto vale $P$ con $t = 1$?

In [ ]:
W3 = ["000", "001", "010", "011", "100", "101"]
W4 = []   # TU CÓDIGO AQUÍ (q3 es el bit de la izquierda)

if W4:
    print("P(t=1) =", simulated_success(grover_circuit(phase_oracle(W4), t=1), W4))

<details>
<summary><b>Ver solución</b></summary>

```python
W4 = ["0" + w for w in W3]
```
$\sin^2\theta = 6/16 \Rightarrow \theta \approx 37.8°$ y $P(1) \approx 0.84$. Duplicar $N$ baja la fracción de marcados por
debajo de 1/2 y Grover vuelve a funcionar.

</details>

---
# Parte 6 · Playground: estados marcados con el oracle builder de IBM

El tutorial de Grover de IBM Quantum construye el oracle con esta función. Léela con calma: **es exactamente lo que
hiciste en la Parte 1**.

| Línea | Qué hace | Dónde lo vimos |
|---|---|---|
| `rev_target = target[::-1]` | invierte el string para leerlo desde el qubit 0 | 1.4, orden de bits |
| `zero_inds = [...]` | qubits donde el estado tiene un 0 | 1.5, `zero_indices` |
| `qc.x(zero_inds)` antes y después | controles abiertos | 1.5 |
| `MCMTGate(ZGate(), num_qubits - 1, 1)` | la MCZ (en los dibujos, `1-Z`) | 1.3 |
| un bloque por estado | varios marcados = producto de reflexiones | 2.2 |

In [ ]:
def grover_oracle(marked_states):
    """Oracle de Grover para varios estados marcados (tutorial de IBM Quantum).

    Se asume que todos los estados tienen el mismo número de bits.
    """
    if not isinstance(marked_states, list):
        marked_states = [marked_states]
    # Número de qubits del circuito
    num_qubits = len(marked_states[0])

    qc = QuantumCircuit(num_qubits)
    # Marcar cada estado de la lista
    for target in marked_states:
        # Invertir el string para seguir el orden de bits de Qiskit
        rev_target = target[::-1]
        # Índices de los '0' del string
        zero_inds = [ind for ind in range(num_qubits) if rev_target.startswith("0", ind)]
        # Z multicontrolada con X antes y después donde hay un '0' (controles abiertos)
        if zero_inds:
            qc.x(zero_inds)
        qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), inplace=True)
        if zero_inds:
            qc.x(zero_inds)
    return qc

IBM completa el algoritmo con `grover_operator(oracle)` de Qiskit (oracle + difusor en un solo bloque) y este número
de iteraciones:

```python
optimal_num_iterations = math.floor(math.pi / (4 * math.asin(math.sqrt(len(marked_states) / 2**grover_op.num_qubits))))
```

Que es nuestra `optimal_t(N, M)`. Comprobemos que todo coincide con lo que construimos:

In [ ]:
W = ["011", "100"]
print("¿grover_oracle(IBM) == phase_oracle?", Operator(grover_oracle(W)) == Operator(phase_oracle(W)))

ibm = QuantumCircuit(3)
ibm.h(range(3))
ibm.compose(grover_operator(grover_oracle(W)).power(optimal_t(8, 2)), inplace=True)
nuestro = grover_circuit(phase_oracle(W), optimal_t(8, 2))
print("¿mismo estado final (salvo fase global)?", statevector(ibm).equiv(statevector(nuestro)))
show(grover_operator(grover_oracle(W)).decompose(), fold=-1)

### El playground

Escribe los estados marcados separados por comas, elige $t$ (o marca **t óptimo**) y mira:

- el oracle que construye el builder de IBM y las fases que pone,
- el histograma de 2000 shots en Aer,
- la curva $P(t)$ con tu $t$ resaltado.

Si los controles no aparecen (algunos editores no muestran widgets), usa la celda de abajo, que llama a
`playground(...)` directamente.

**Experimentos sugeridos**

1. `10110` (n = 5, un marcado). Mueve $t$ de 0 a 15: ¿dónde está el máximo? ¿y el segundo?
2. `0000, 1111` y luego `0000, 0101, 1010, 1111`. ¿Cuánto cambia $t_{\text{opt}}$?
3. La mitad de los estados de 3 qubits: `000, 001, 010, 011`. Prueba varios $t$.
4. Seis de ocho: `000, 001, 010, 011, 100, 101` con $t = 1$.
5. Un estado repetido: `101, 101`. ¿Qué marcó el oracle?
6. Deja la caja vacía (con `n` = 3). El builder de IBM no acepta una lista vacía: ¿por qué?
7. Mezcla longitudes, por ejemplo `01, 110`. ¿Qué debería pasar?

In [ ]:
import ipywidgets as widgets


def _parse(texto):
    return [s.strip() for s in texto.replace(";", ",").split(",") if s.strip()]


def playground(marcados, t=None, n=None, shots=2000):
    """Grover con el oracle builder de IBM. t=None usa el t óptimo."""
    marcados = _parse(marcados) if isinstance(marcados, str) else list(marcados)
    if any(set(s) - {"0", "1"} for s in marcados):
        print("Los estados solo pueden tener 0 y 1."); return
    if len({len(s) for s in marcados}) > 1:
        print("Todos los estados deben tener el mismo número de bits: el builder de IBM usa el largo del primero")
        print("para decidir cuántos qubits tiene el circuito."); return
    if marcados:
        n = len(marcados[0])
        oracle = grover_oracle(marcados)
    else:
        n = n or 3
        print(f"Lista vacía: grover_oracle([]) fallaría en marked_states[0]. Usamos un oracle vacío de {n} qubits (M = 0).")
        oracle = QuantumCircuit(n)
    if n > 8:
        print("Usa a lo más 8 qubits para que la simulación sea rápida."); return

    N = 2**n
    fase_menos = [format(i, f"0{n}b") for i, d in enumerate(np.diag(Operator(oracle).data)) if np.isclose(d, -1)]
    M = len(fase_menos)
    t = optimal_t(N, M) if t is None else t
    repetidos = sorted({s for s in marcados if marcados.count(s) > 1})

    print(f"N = {N}, estados escritos = {len(marcados)}, marcados de verdad (fase -1) = {M}")
    if repetidos:
        print(f"⚠️  Repetidos: {repetidos}. Cada par de repeticiones se cancela (Z·Z = I).")
    print(f"θ = {math.degrees(theta(N, M)):.1f}°   t_opt = {optimal_t(N, M)}   t usado = {t}   "
          f"P teórica = {success_probability(N, M, t):.3f}")

    if marcados and len(marcados) <= 4 and n <= 5:
        show(oracle, fold=-1)

    qc = QuantumCircuit(n)
    qc.h(range(n))
    if t:
        qc.compose(grover_operator(oracle).power(t), inplace=True)
    c = counts(qc, shots=shots, seed=1)
    medida = sum(v for k, v in c.items() if k in fase_menos) / shots
    print(f"P medida ({shots} shots) = {medida:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), gridspec_kw={"width_ratios": [3, 2]})
    estados = [format(i, f"0{n}b") for i in range(N)]
    axes[0].bar(range(N), [c.get(s, 0) for s in estados], width=0.7,
                color=[MARKED if s in fase_menos else NORMAL for s in estados])
    axes[0].set_xticks(range(N))
    axes[0].set_xticklabels(estados, rotation=90 if n > 3 else 0, fontsize=7 if n > 5 else 9)
    axes[0].set_ylabel("conteos")
    axes[0].set_title(f"Histograma con t = {t} (naranja = marcado)", fontsize=11)
    _style(axes[0])
    plot_success_vs_t(N, M, t_max=max(10, 3 * optimal_t(N, M) + 2, t + 2), highlight_t=t, ax=axes[1])
    plt.tight_layout(); plt.show()


caja = widgets.Text(value="10110", description="marcados", continuous_update=False,
                    layout=widgets.Layout(width="420px"))
n_vacio = widgets.IntSlider(value=3, min=2, max=8, description="n si vacío")
t_slider = widgets.IntSlider(value=1, min=0, max=25, description="t", continuous_update=False)
usar_opt = widgets.Checkbox(value=True, description="t óptimo")

ui = widgets.VBox([caja, widgets.HBox([t_slider, usar_opt]), n_vacio])
out = widgets.interactive_output(
    lambda marcados, t, opt, n: playground(marcados, None if opt else t, n),
    {"marcados": caja, "t": t_slider, "opt": usar_opt, "n": n_vacio},
)
display(ui, out)

In [ ]:
# Versión sin widgets: cambia los argumentos y ejecuta
playground("0000, 0101, 1010, 1111", t=None)

In [ ]:
playground("000, 001, 010, 011, 100, 101", t=1)

**Preguntas del playground**

- ¿Por qué el builder de IBM necesita que todos los estados tengan el mismo largo?
- `grover_oracle([])` falla. ¿Qué circuito debería devolver para $M = 0$? (Parte 2.3)
- Con `101, 101` el oracle no marca nada. ¿Debería el builder eliminar duplicados? ¿Qué harías tú?
- Compara `grover_operator(oracle)` con `grover_iteration(oracle)` de la Parte 3 usando `Operator(...).equiv(...)`.

<details>
<summary><b>Ver solución</b></summary>

- El número de qubits se toma de `len(marked_states[0])`; un estado más corto daría un `zero_inds` incompleto
  (o un error) y marcaría otra cosa.
- Para $M = 0$ el oracle correcto es la identidad: `QuantumCircuit(n)` vacío. Hay que pasarle `n` aparte, como hace
  `phase_oracle([], n=...)`.
- Lo razonable es usar `set(marked_states)`: un estado marcado lo está una vez. Con la lista tal cual, cada par de
  repeticiones se cancela.
- `Operator(grover_operator(o)).equiv(Operator(grover_iteration(o)))` da `True`: son la misma iteración salvo fase global.

</details>

---
# Parte 7 · Reto: oracles de verdad

Hasta ahora sabíamos qué estados marcar. En un problema real **no**: el oracle debe *calcular* si $x$ cumple una condición.

### 7.1 El problema: ¿a quién invito a la fiesta?

Cuatro amigos: **A**na ($q_0$), **B**eto ($q_1$), **C**arla ($q_2$) y **D**ani ($q_3$). $q_i = 1$ significa "va".

| Cláusula | Significado |
|---|---|
| $A \lor B$ | va al menos Ana o Beto |
| $\lnot A \lor C$ | si va Ana, va Carla |
| $\lnot B \lor \lnot D$ | Beto y Dani no van juntos |
| $C \lor D$ | va Carla o Dani |
| $\lnot C \lor \lnot D$ | Carla y Dani no van juntos |

### 7.2 Construcción del oracle

Para cada cláusula $(\ell_1 \lor \ell_2)$ usamos una ancilla que empieza en $|0\rangle$:

1. La cláusula es **falsa** solo si los dos literales son falsos. Con X en los literales positivos, una MCX (Toffoli)
   pone la ancilla en 1 cuando la cláusula es falsa.
2. Una X en la ancilla la invierte: ahora vale 1 si la cláusula es **verdadera**.

Una **MCZ sobre todas las ancillas** pone el $-1$ cuando todas se cumplen. Al final hay que **deshacer** (*uncompute*)
el cálculo de las ancillas para devolverlas a $|0\rangle$; si no, quedan entrelazadas con $|x\rangle$ y la interferencia
se rompe.

In [ ]:
A, B, C, D = 0, 1, 2, 3
CLAUSULAS = [
    [(A, True), (B, True)],
    [(A, False), (C, True)],
    [(B, False), (D, False)],
    [(C, True), (D, True)],
    [(C, False), (D, False)],
]


def calcula_clausula(qc, clausula, anc):
    """Deja en `anc` el valor (l1 OR l2 OR ...) de la cláusula."""
    positivos = [q for q, pos in clausula if pos]
    if positivos:
        qc.x(positivos)
    qc.append(MCXGate(len(clausula)), [q for q, _ in clausula] + [anc])   # anc = 1 si la cláusula es falsa
    if positivos:
        qc.x(positivos)
    qc.x(anc)                                                              # anc = 1 si es verdadera


def oracle_sat(clausulas, n):
    m = len(clausulas)
    ancillas = list(range(n, n + m))
    calcula = QuantumCircuit(n + m)
    for cl, anc in zip(clausulas, ancillas):
        calcula_clausula(calcula, cl, anc)
    qc = QuantumCircuit(n + m, name="O_SAT")
    qc.compose(calcula, inplace=True)
    qc.compose(mcz(m), qubits=ancillas, inplace=True)
    qc.compose(calcula.inverse(), inplace=True)   # uncompute
    return qc


oracle = oracle_sat(CLAUSULAS, n=4)
oracle.draw("mpl", fold=-1)

Para **verificar** (solo para comprobar, ¡es hacer trampa!) resolvemos por fuerza bruta:

In [ ]:
def cumple(x, clausulas):
    bits = [(x >> i) & 1 for i in range(4)]
    return all(any(bits[q] == pos for q, pos in cl) for cl in clausulas)

soluciones = marked_from_function(lambda x: cumple(x, CLAUSULAS), 4)
print("soluciones:", soluciones)
check_oracle(oracle, soluciones, n=4);

Ahora, Grover. Los 4 primeros qubits son de búsqueda; las 5 ancillas no se tocan con el difusor.

In [ ]:
M = len(soluciones)
t = optimal_t(16, M)
print(f"M = {M}, t_opt = {t}, P teórica = {success_probability(16, M, t):.3f}")
histograma(counts(grover_circuit(oracle, t, n=4, measure=True), shots=2000, seed=11))

**Pregunta.** Traduce las soluciones a listas de invitados (recuerda: `"dcba"`, Ana es el bit de la derecha).

**Pregunta.** ¿Qué pasa si quitas la línea del *uncompute*? Copia `oracle_sat`, bórrala y corre `check_oracle` y Grover.

### Reto 1: una condición más

Beto se pone exigente: **"si voy yo, va Ana"** ($\lnot B \lor A$). Agrega la cláusula, construye el oracle y encuentra
las nuevas soluciones. Fíjate en que cambió $M$, y con él $t_{\text{opt}}$.

In [ ]:
CLAUSULAS_2 = CLAUSULAS + [
    # TU CÓDIGO AQUÍ: la nueva cláusula
]
oracle_2 = oracle_sat(CLAUSULAS_2, n=4)
soluciones_2 = marked_from_function(lambda x: cumple(x, CLAUSULAS_2), 4)
check_oracle(oracle_2, soluciones_2, n=4);

# TU CÓDIGO AQUÍ: corre Grover con el t adecuado y grafica

<details>
<summary><b>Ver solución</b></summary>

```python
CLAUSULAS_2 = CLAUSULAS + [[(B, False), (A, True)]]
oracle_2 = oracle_sat(CLAUSULAS_2, n=4)
soluciones_2 = marked_from_function(lambda x: cumple(x, CLAUSULAS_2), 4)   # ['0101', '0111']
t2 = optimal_t(16, len(soluciones_2))                                        # 2
histograma(counts(grover_circuit(oracle_2, t2, n=4, measure=True), shots=2000))
```
Ahora $M = 2$, así que hacen falta 2 iteraciones en lugar de 1.

</details>

### Reto 2: el oracle secreto

`secret_oracle(n=5, seed=...)` marca entre 1 y 3 estados de 5 qubits escogidos al azar, envueltos en una sola compuerta
opaca. Usa **tu número de lista** como `seed` y encuentra **todos** los estados marcados usando solo Grover y mediciones.

- No sabes $M$: prueba varios $t$ y mira cómo cambian los histogramas (sección 5.8).
- Con $N = 32$: $t_{\text{opt}} = 4$ si $M = 1$, $3$ si $M = 2$ y $2$ si $M = 3$.
- Un estado marcado aparece mucho más que $1/32$ de las veces.

Cuando creas que los tienes, verifica con `check_secret`.

In [ ]:
def _secret_marked(n, seed):
    rng = random.Random(seed)
    return sorted(format(x, f"0{n}b") for x in rng.sample(range(2**n), rng.randint(1, 3)))


def secret_oracle(n=5, seed=0):
    """Oracle caja negra. (¡No hagas trampa mirando por dentro!)"""
    gate = phase_oracle(_secret_marked(n, seed), n=n).to_gate(label="Oracle secreto")
    qc = QuantumCircuit(n)
    qc.append(gate, range(n))
    return qc


def check_secret(guess, n=5, seed=0):
    secret, guess = set(_secret_marked(n, seed)), set(guess)
    if guess == secret:
        print(f"✅ ¡Encontrados! El oracle secreto marcaba {sorted(secret)} (M = {len(secret)}).")
        return True
    right = guess & secret
    print(f"❌ Aún no. Aciertos: {len(right)} de tu lista de {len(guess)}.",
          "Te faltan estados." if len(right) == len(guess) else "Hay estados que no están marcados.")
    return False


SEED = 0          # cambia esto por tu número
caja_negra = secret_oracle(n=5, seed=SEED)
caja_negra.draw("mpl")

In [ ]:
# TU CÓDIGO AQUÍ: explora con distintos t
for t in [2, 3, 4]:
    c = counts(grover_circuit(caja_negra, t, measure=True), shots=1000, seed=t)
    print(f"t = {t}:", sorted(c.items(), key=lambda kv: -kv[1])[:5])

In [ ]:
mi_respuesta = []   # TU RESPUESTA AQUÍ, p. ej. ["00101", "11000"]
check_secret(mi_respuesta, n=5, seed=SEED);

<details>
<summary><b>Ver solución</b></summary>

Para cada $t$ quédate con los estados que salen mucho más que $1000/32 \approx 31$ veces y comprueba que el número de
picos sea consistente con el $t$ que mejor funciona. También puedes construir tu propio `phase_oracle(candidatos)`
y comparar su efecto con el de la caja.

</details>

---
## Para ir más allá

- Reescribe `oracle_sat` usando el **oracle de bit-flip** con una ancilla en $|-\rangle$ en vez de la MCZ final.
- Cuenta las compuertas de dos qubits tras `transpile(qc, basis_gates=["cx", "u"])` para distintos $t$. ¿Cómo crece el costo?
- Codifica otro problema pequeño (coloreado de un triángulo, un sudoku 2×2...) y resuélvelo con Grover.